## Set Up


In [ ]:
!pip install yfinance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()


## Data Analysis

### S&P 500 Sectors

In [ ]:
import urllib
from urllib.request import urlopen
from bs4 import BeautifulSoup

u = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = urlopen(u).read()
soup = BeautifulSoup(html)

table = soup.find_all('table')[0]

d = []
for row in table.find_all('tr'):
    columns = row.find_all('td')
    r = []
    if columns:
        for column in columns:
            r.append(column.get_text())
        d.append(r)

df = pd.DataFrame(d)
df = df.rename(columns = {0: 'Symbol', 1: 'Security', 2:'SEC', 3: 'Sector', 4: 'Sub Industry', 5: 'HQ', 6: 'Date Added', 7: 'CIK', 8: 'Founded'})

df = df[['Symbol', 'Security', 'Sector']]

for c in df.columns:
    df[c] = df[c].apply(lambda x: x.replace('\n', ''))

In [ ]:
import urllib.request

user_agent = 'Mozilla/5.0 (Windows; U; Windows NT 5.1; en-US; rv:1.9.0.7) Gecko/2009021910 Firefox/3.0.7'

url = "https://www.slickcharts.com/sp500"
headers={'User-Agent':user_agent,} 

request = urllib.request.Request(url,None,headers) #The assembled request
response = urllib.request.urlopen(request)
data = response.read() # The data u need

soup = BeautifulSoup(data)

table = soup.find_all('table')[0]

d = []
for row in table.find_all('tr'):
    columns = row.find_all('td')
    r = []
    if columns:
        for column in columns:
            r.append(column.get_text())
        d.append(r)

weights = pd.DataFrame(d)

weights = weights.rename(columns = {2: 'Symbol', 3: 'Weight'})[['Symbol', 'Weight']]

In [ ]:
final_df = df.merge(weights, on = ['Symbol']).groupby('Sector')
final_df = final_df.apply(lambda x: x.sort_values('Weight', ascending = False))
final_df = final_df.reset_index(drop = True)

final_df['Weight'] = final_df['Weight'].astype(float)

final_df_10 = final_df.groupby('Sector').head(10)

# top_10_df = pd.DataFrame()
# for s in final_df_10['Sector'].unique():
#     if top_10_df.empty: top_10_df = final_df_10[final_df_10['Sector'] == s]
#     else: top_10_df = top_10_df.append(final_df_10[final_df_10['Sector'] == s])

In [ ]:
final_df.groupby(['Sector']).sum().plot.bar();

In [ ]:
final_df.groupby(['Sector']).size().plot.bar();

In [ ]:
(final_df.groupby(['Sector']).sum()/pd.DataFrame(final_df.groupby(['Sector']).size(), columns = ['Weight'])).plot.bar()

In [ ]:
final_df.groupby(['Sector']).sum()

In [ ]:
final_df = final_df.sort_values('Weight', ascending = False).reset_index(drop = True)

In [ ]:
final_df[final_df['Sector'] == 'Utilities']

In [ ]:
final_df.head()

In [ ]:
tickers = fdf.sort_values(by = 'Factor', ascending = False).head(10)['Symbol'].apply(lambda x: x.replace('.', '-')).tolist()

In [ ]:
from IPython.display import clear_output

start_date = '2019-08-01'
end_date = date.today()

l = {}
err = []
for ticker in tickers:
    try:
        temp_df = pdr.get_data_yahoo(ticker, start = start_date, end = end_date).reset_index().dropna()['Close']
        l[ticker] = (temp_df.iloc[-1] - temp_df.iloc[0])/temp_df.iloc[0]
        clear_output()
    except Exception as e:
        err.append(ticker)

In [ ]:
err

In [ ]:
growth_df = pd.DataFrame(l.items(), columns = ['Symbol', 'One_Year_Growth'])
growth_df['Symbol'] = growth_df['Symbol'].apply(lambda x: x.replace('-', '.'))

In [ ]:
fdf = pd.merge(final_df, growth_df)
fdf['One_Year_Growth'] = fdf['One_Year_Growth']/100

In [ ]:
fdf['Factor'] = (fdf['One_Year_Growth'] + fdf['Weight'])/2

In [ ]:
growth_df#['One_Year_Growth'].mean()

In [ ]:
fdf.sort_values(by = 'Factor', ascending = False).head(10)

In [ ]:
fdf.sort_values(by = 'One_Year_Growth', ascending = False).head(10)

In [ ]:
l1 = []
l2 = []
l3 = []
for i in range(0, 20):
    l1.append(fdf.sort_values(by = 'Weight', ascending = False).head(i)['One_Year_Growth'].mean())
    l2.append(fdf.sort_values(by = 'One_Year_Growth', ascending = False).head(i)['One_Year_Growth'].mean())
    l3.append(fdf.sort_values(by = 'Factor', ascending = False).head(i)['One_Year_Growth'].mean())

plt.figure(figsize = (20, 8))
plt.plot(range(len(l1)), l1, label = 'Weight')
plt.plot(range(len(l2)), l2, label = 'One_Year_Growth')
plt.plot(range(len(l3)), l3, label = 'Factor')
plt.legend()

In [ ]:
l1 = []
l2 = []
l3 = []
for i in range(0, 20):
    l1.append(fdf.sort_values(by = 'Weight', ascending = False).head(i)['One_Year_Growth'].mean())
    l2.append(fdf.sort_values(by = 'One_Year_Growth', ascending = False).head(i)['One_Year_Growth'].mean())
    l3.append(fdf.sort_values(by = 'Factor', ascending = False).head(i)['One_Year_Growth'].mean())

plt.figure(figsize = (20, 8))
plt.plot(range(len(l1)), l1, label = 'Weight')
plt.plot(range(len(l2)), l2, label = 'One_Year_Growth')
plt.plot(range(len(l3)), l3, label = 'Factor')
plt.legend()